In [59]:
import numpy as np
import pandas as pd
import os
from glob import glob
import json
from datetime import datetime

In [60]:
arr = []
files = glob(os.path.join('json', '*.json'))

for file in files:
  if ("Streaming_History_Audio" in file):  
    with open (file, 'r') as f:
      try:
        contents = json.load(f)
        arr.extend(contents)
      except Exception as e:
        print(e)

select = [('ts','ts'), ('ms_played', 'ms_played'), ('reason_start', 'reason_start'), 
  ('reason_end', 'reason_end'), 
  ('master_metadata_track_name', 'track'), ('master_metadata_album_artist_name', 'artist'),
  ('master_metadata_album_album_name', 'album'), ('spotify_track_uri','uri'), ('conn_country', 'conn_country')]

data = np.array([[i[k[0]] for k in select] for i in arr])
df = pd.DataFrame(data)
df.columns = [k[1] for k in select]
streams = df[df['ms_played'] >= 30000].copy()
streams['ts'] = pd.to_datetime(streams['ts']).dt.tz_convert('America/Vancouver')

streams = streams.set_index('ts').sort_index()
streams['date'] = streams.index.strftime('%Y-%m-%d')

streams.to_csv('csv/streams.csv')

YEAR = "2025"
dated_streams = streams.loc[YEAR]
dated_streams.to_csv('csv/dated_streams.csv')

dated_months = pd.date_range(start=YEAR + "-01", end=YEAR + "-12", freq='MS')

In [61]:
monthly_top_artists = pd.DataFrame([])
months = pd.period_range(start="2018-01", end=streams.index[-1].strftime('%Y-%m'), freq='M')

for m in months:
  pivot_table = streams.loc[m.strftime('%Y-%m')].pivot_table(
      index='artist',
      values=['ms_played'],
      aggfunc={'ms_played':'sum'},
  ).sort_values(by='ms_played',ascending=False).head(50)
  monthly_top_artists[m] = pd.Series(pivot_table.index)

monthly_top_artists.index = list(range(1,51))
monthly_top_artists.to_csv('csv/monthly_top_artists.csv')

top_artists_pos = {}
for m, c in monthly_top_artists.iteritems():
  for index, artist in c.items():
    if artist not in top_artists_pos:
      top_artists_pos[artist] = {}
    top_artists_pos[artist][m.strftime('%Y-%m')] = index

top_artists_pos = pd.DataFrame.from_dict(top_artists_pos, orient='index').filter(like=YEAR)
top_artists_pos = top_artists_pos.loc[(top_artists_pos <= 10).any(axis=1)]
top_artists_pos.to_csv('csv/top_artists_pos.csv')

In [62]:
top_artists = dated_streams.pivot_table(
    index='artist',
    values=['ms_played', 'uri', 'track', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'track': 'nunique', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_artists['h'] = round(top_artists['ms_played'] / 1000 / 60 / 60, 1)
top_artists.to_csv('csv/top_artists.csv')

top_tracks = dated_streams.pivot_table(
    index='track',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_tracks['h'] = round(top_tracks['ms_played'] / 1000 / 60 / 60, 1)
top_tracks.to_csv('csv/top_tracks.csv')

In [63]:
corrections = {   # based on top 50 albums for 2025, correcting to properly aggregate
  'eternal sunshine deluxe: brighter days ahead': 'eternal sunshine',
  'Nettles': 'Willoughby Tucker, I\'ll Always Love You',
  'Sunshine & Rain...': 'Sincerely',
  'emails i can’t send fwd:': 'emails i can’t send',
  'GUTS (spilled)': 'GUTS',
  'Kansas Anymore (The Longest Goodbye)': 'Kansas Anymore',
  'Headphones On': 'Addison',
}

for k, v in corrections.items():
  dated_streams.loc[dated_streams['album'] == k, 'album'] = v

top_albums = dated_streams.pivot_table(
    index=['album', 'artist'],
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_albums['h'] = round(top_albums['ms_played'] / 1000 / 60 / 60, 1)
top_albums.to_csv('csv/top_albums.csv')

In [64]:
[df["ms_played"].sum(),
streams.agg({"ms_played": "sum", "uri": "count", "artist": "nunique", "track": "nunique"})]

[11454297633,
 ms_played    11204603920
 uri                63702
 artist              1556
 track               5416
 dtype: int64]

In [65]:
tod_matrix = [{} for i in range(7)]
for d in range(7):
  for t in ["early morning", "morning", "midday", "afternoon", "evening", "night", "late night"]:
    tod_matrix[d][t] = 0

for ts, ms_played in zip(dated_streams.index, dated_streams["ms_played"]):
  tod = "late night"
  if ts.hour < 6: tod = "early morning"
  elif ts.hour < 10: tod = "morning"
  elif ts.hour < 13: tod = "midday"
  elif ts.hour < 16: tod = "afternoon"
  elif ts.hour < 19: tod = "evening"
  elif ts.hour < 22: tod = "night"
  tod_matrix[ts.day_of_week][tod] += ms_played

tod_matrix = pd.DataFrame(tod_matrix)
tod_matrix.index = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
tod_matrix = tod_matrix.T / 1000 / 60 / 60
tod_matrix.insert(loc=0, column='Range', value=["12AM-06AM", "06AM-10AM", "10AM-01PM", "01PM-04PM", "04PM-07PM", "07PM-10PM", "10PM-12PM"])

tod_matrix.to_csv('csv/tod_matrix.csv')

hour_matrix = [[0 for j in range(24)] for i in range (7)]
for ts, ms_played in zip(dated_streams.index, dated_streams["ms_played"]):
  hour_matrix[ts.day_of_week][ts.hour] += ms_played

hour_matrix = pd.DataFrame(hour_matrix) / 1000 / 60 / 60
hour_matrix.index = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
hour_matrix.to_csv('csv/hour_matrix.csv')

day_matrix = [[0 for j in range(53)] for i in range(7)]
for ts, ms_played in zip(dated_streams.index, dated_streams["ms_played"]):
  year, week, day = ts.isocalendar()
  if (year > int(YEAR)): week = 53
  day_matrix[day-1][week-1] += ms_played

day_matrix = pd.DataFrame(day_matrix) / 1000 / 60 / 60
day_matrix.to_csv('csv/day_matrix.csv')

In [66]:
mask = (dated_streams["artist"].isin(top_artists.head(5).index)) & (dated_streams['reason_start'] != "appload")
top_five = dated_streams[mask].pivot_table(
    index=['artist','track'],
    values=['uri'],
    aggfunc={'uri': 'count'}
).sort_values(by='uri',ascending=False)

top_five = top_five.groupby('artist').apply(lambda x: x.nlargest(5, 'uri'))
top_five.to_csv('csv/top_five.csv')

In [67]:
monthly_stats = streams[streams["reason_start"] != "appload"]
monthly_stats = monthly_stats.groupby(monthly_stats.index.strftime('%Y-%m')).agg({'ms_played': 'sum', 'uri': 'count', 'track': 'nunique', 'artist': 'nunique'})
monthly_stats.to_csv('csv/monthly_stats.csv')

In [68]:
monthly_top_tracks = pd.DataFrame([])
months = pd.period_range(start="2018-01", end=streams.index[-1].strftime('%Y-%m'), freq='M')

for m in months:
  pivot_table = streams.loc[m.strftime('%Y-%m')].pivot_table(
      index='track',
      values=['ms_played', 'uri'],
      aggfunc={'ms_played':'sum', 'uri': 'count'},
  ).sort_values(by='uri',ascending=False).head(50)
  monthly_top_tracks[m] = pd.Series(pivot_table.index)

monthly_top_tracks.index = list(range(1,51))
monthly_top_tracks.to_csv('csv/monthly_top_tracks.csv')